In [ ]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

In [ ]:
import os

DATA_PATH = "/kaggle/input/datasets/harshithabhaskaran/echonet-final-dataset-001"

IMG_PATH = os.path.join(DATA_PATH, "images")
MASK_PATH = os.path.join(DATA_PATH, "masks")

print(len(os.listdir(IMG_PATH)), "images")
print(len(os.listdir(MASK_PATH)), "masks")

In [ ]:
import cv2
import numpy as np
from torch.utils.data import Dataset

class EchoDataset(Dataset):
    def __init__(self, img_dir, mask_dir):
        self.img_dir = img_dir
        self.mask_dir = mask_dir
        self.images = sorted(os.listdir(img_dir))

    def __len__(self):
        return len(self.images)

    def __getitem__(self, idx):
        img_name = self.images[idx]

        img_path = os.path.join(self.img_dir, img_name)
        mask_path = os.path.join(self.mask_dir, img_name)

        image = cv2.imread(img_path)
        image = cv2.resize(image, (112, 112))
        image = image / 255.0
        image = np.transpose(image, (2, 0, 1))

        mask = cv2.imread(mask_path, 0)
        mask = cv2.resize(mask, (112, 112))
        mask = (mask > 0).astype(np.float32)
        mask = np.expand_dims(mask, axis=0)

        return image.astype(np.float32), mask

In [ ]:
from torch.utils.data import DataLoader

dataset = EchoDataset(IMG_PATH, MASK_PATH)

train_loader = DataLoader(dataset, batch_size=16, shuffle=True)

print("Dataset ready:", len(dataset))

In [ ]:
import torch
import torch.nn as nn

class UNet(nn.Module):
    def __init__(self):
        super().__init__()

        def block(in_c, out_c):
            return nn.Sequential(
                nn.Conv2d(in_c, out_c, 3, padding=1),
                nn.ReLU(),
                nn.Conv2d(out_c, out_c, 3, padding=1),
                nn.ReLU()
            )

        self.enc1 = block(3, 64)
        self.pool1 = nn.MaxPool2d(2)

        self.enc2 = block(64, 128)
        self.pool2 = nn.MaxPool2d(2)

        self.enc3 = block(128, 256)
        self.pool3 = nn.MaxPool2d(2)

        self.bottleneck = block(256, 512)

        self.up3 = nn.ConvTranspose2d(512, 256, 2, 2)
        self.dec3 = block(512, 256)

        self.up2 = nn.ConvTranspose2d(256, 128, 2, 2)
        self.dec2 = block(256, 128)

        self.up1 = nn.ConvTranspose2d(128, 64, 2, 2)
        self.dec1 = block(128, 64)

        self.final = nn.Conv2d(64, 1, 1)

    def forward(self, x):
        e1 = self.enc1(x)
        e2 = self.enc2(self.pool1(e1))
        e3 = self.enc3(self.pool2(e2))

        b = self.bottleneck(self.pool3(e3))

        d3 = self.up3(b)
        d3 = self.dec3(torch.cat([d3, e3], dim=1))

        d2 = self.up2(d3)
        d2 = self.dec2(torch.cat([d2, e2], dim=1))

        d1 = self.up1(d2)
        d1 = self.dec1(torch.cat([d1, e1], dim=1))

        return torch.sigmoid(self.final(d1))

In [ ]:
model = UNet()
model = model.to("cuda" if torch.cuda.is_available() else "cpu")

In [ ]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

In [ ]:
model = UNet().to(device)

In [ ]:
optimizer = torch.optim.Adam(model.parameters(), lr=1e-4)
criterion = nn.BCEWithLogitsLoss()

In [ ]:
model.eval()

img, mask = dataset[10]  # try different index

img = img.astype(np.float32)
if img.max() > 1:
    img = img / 255.0

img_tensor = torch.tensor(img).unsqueeze(0).to(device)

with torch.no_grad():
    pred = torch.sigmoid(model(img_tensor))[0][0].cpu().numpy()

pred_binary = (pred > 0.5).astype(np.uint8)

import matplotlib.pyplot as plt

plt.figure(figsize=(12,4))

plt.subplot(1,3,1)
plt.title("Image")
plt.imshow(np.transpose(img, (1,2,0)))
plt.axis('off')

plt.subplot(1,3,2)
plt.title("GT")
plt.imshow(mask[0] if len(mask.shape)==3 else mask, cmap='gray')
plt.axis('off')

plt.subplot(1,3,3)
plt.title("Prediction")
plt.imshow(pred_binary, cmap='gray')
plt.axis('off')

plt.show()

In [ ]:
# ==============================
# FULL FIX: MODEL + TRAIN
# ==============================
import torch
import torch.nn as nn
import numpy as np
import matplotlib.pyplot as plt

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

# ==============================
# PROPER U-NET
# ==============================
class DoubleConv(nn.Module):
    def __init__(self, in_c, out_c):
        super().__init__()
        self.net = nn.Sequential(
            nn.Conv2d(in_c, out_c, 3, padding=1),
            nn.ReLU(inplace=True),
            nn.Conv2d(out_c, out_c, 3, padding=1),
            nn.ReLU(inplace=True),
        )
    def forward(self, x):
        return self.net(x)

class UNet(nn.Module):
    def __init__(self):
        super().__init__()

        self.down1 = DoubleConv(3, 64)
        self.pool1 = nn.MaxPool2d(2)

        self.down2 = DoubleConv(64, 128)
        self.pool2 = nn.MaxPool2d(2)

        self.middle = DoubleConv(128, 256)

        self.up1 = nn.ConvTranspose2d(256, 128, 2, stride=2)
        self.conv1 = DoubleConv(256, 128)

        self.up2 = nn.ConvTranspose2d(128, 64, 2, stride=2)
        self.conv2 = DoubleConv(128, 64)

        self.out = nn.Conv2d(64, 1, 1)

    def forward(self, x):
        d1 = self.down1(x)
        p1 = self.pool1(d1)

        d2 = self.down2(p1)
        p2 = self.pool2(d2)

        m = self.middle(p2)

        u1 = self.up1(m)
        u1 = torch.cat([u1, d2], dim=1)
        u1 = self.conv1(u1)

        u2 = self.up2(u1)
        u2 = torch.cat([u2, d1], dim=1)
        u2 = self.conv2(u2)

        return self.out(u2)

# ==============================
# INIT MODEL
# ==============================
model = UNet().to(device)

criterion = nn.BCEWithLogitsLoss()
optimizer = torch.optim.Adam(model.parameters(), lr=1e-3)

# ==============================
# TRAIN
# ==============================
EPOCHS = 3

for epoch in range(EPOCHS):
    model.train()
    total_loss = 0

    print(f"\n🚀 Epoch {epoch+1}")

    for i, (images, masks) in enumerate(train_loader):
        images = images.float().to(device)
        masks = masks.float().to(device)

        if images.max() > 1:
            images = images / 255.0

        if masks.dim() == 3:
            masks = masks.unsqueeze(1)

        preds = model(images)
        loss = criterion(preds, masks)

        optimizer.zero_grad()
        loss.backward()
        optimizer.step()

        total_loss += loss.item()

        if i % 10 == 0:
            print(f"Batch {i} Loss: {loss.item():.4f}")

    print("Epoch Loss:", total_loss)

# ==============================
# TEST
# ==============================
model.eval()

img, mask = dataset[0]
img = img.astype(np.float32)
if img.max() > 1:
    img = img / 255.0

img_tensor = torch.tensor(img).unsqueeze(0).to(device)

with torch.no_grad():
    pred = torch.sigmoid(model(img_tensor))[0][0].cpu().numpy()

print("\n🔍 Prediction stats:", pred.min(), pred.max())

plt.figure(figsize=(12,4))

plt.subplot(1,3,1)
plt.title("Image")
plt.imshow(np.transpose(img, (1,2,0)))
plt.axis('off')

plt.subplot(1,3,2)
plt.title("GT")
plt.imshow(mask[0] if len(mask.shape)==3 else mask, cmap='gray')
plt.axis('off')

plt.subplot(1,3,3)
plt.title("Prediction")
plt.imshow(pred > 0.5, cmap='gray')
plt.axis('off')

plt.show()

In [ ]:
import numpy as np

def compute_volume(mask, pixel_spacing=0.1):
    volume = 0
    h = pixel_spacing

    for row in mask:
        width = np.sum(row > 0)
        if width > 0:
            r = (width * pixel_spacing) / 2
            volume += np.pi * r * r * h

    return volume

def volume_curve(masks):
    return np.array([compute_volume(m) for m in masks])

In [ ]:
from scipy.ndimage import gaussian_filter1d
from scipy.signal import find_peaks

def get_cardiac_phases(volumes):
    smooth = gaussian_filter1d(volumes, sigma=2)

    peaks, _ = find_peaks(smooth)       # ED (max)
    valleys, _ = find_peaks(-smooth)    # ES (min)

    return smooth, peaks, valleys

In [ ]:
def compute_ef(volumes, peaks, valleys):
    ef_values = []

    for ed, es in zip(peaks, valleys):
        EDV = volumes[ed]
        ESV = volumes[es]

        if EDV > 0:
            ef = (EDV - ESV) / EDV * 100
            ef_values.append(ef)

    return ef_values

In [ ]:
import torch.nn.functional as F

class DiceLoss(nn.Module):
    def forward(self, preds, targets, smooth=1e-6):
        preds = torch.sigmoid(preds)

        preds = preds.view(-1)
        targets = targets.view(-1)

        intersection = (preds * targets).sum()

        dice = (2. * intersection + smooth) / (preds.sum() + targets.sum() + smooth)
        return 1 - dice

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from scipy.ndimage import gaussian_filter1d
from scipy.signal import find_peaks

# ------------------------------
# Volume
# ------------------------------
def compute_volume(mask, pixel_spacing=0.1):
    volume = 0
    h = pixel_spacing

    for row in mask:
        width = np.sum(row > 0)
        if width > 0:
            r = (width * pixel_spacing) / 2
            volume += np.pi * r * r * h

    return volume

def volume_curve(masks):
    print("👉 Computing volumes...")
    vols = [compute_volume(m) for m in masks]
    print("Sample volumes:", vols[:5])
    return np.array(vols)

# ------------------------------
# Cardiac phases
# ------------------------------
def get_cardiac_phases(volumes):
    print("👉 Smoothing volumes...")
    smooth = gaussian_filter1d(volumes, sigma=2)

    peaks, _ = find_peaks(smooth)
    valleys, _ = find_peaks(-smooth)

    print("Peaks:", peaks)
    print("Valleys:", valleys)

    return smooth, peaks, valleys

# ------------------------------
# EF
# ------------------------------
def compute_ef(volumes, peaks, valleys):
    ef_values = []

    for ed, es in zip(peaks, valleys):
        EDV = volumes[ed]
        ESV = volumes[es]

        if EDV > 0:
            ef = (EDV - ESV) / EDV * 100
            ef_values.append(ef)

    return ef_values

# ------------------------------
# FULL PIPELINE
# ------------------------------
def run_ef_pipeline(masks):
    print("\n🚀 Running EF pipeline...")

    if len(masks) == 0:
        print("❌ No masks provided")
        return

    vols = volume_curve(masks)

    if len(vols) < 5:
        print("❌ Not enough frames for EF")
        return

    smooth, peaks, valleys = get_cardiac_phases(vols)

    ef_values = compute_ef(smooth, peaks, valleys)

    print("\n📊 EF values:", ef_values)

    if len(ef_values) == 0:
        print("⚠️ No EF detected (peaks/valleys issue)")
    else:
        avg = np.mean(ef_values)
        print("Average EF:", avg)

        if avg < 40:
            print("⚠️ Dysfunction")
        else:
            print("✅ Normal")

    # Plot
    plt.figure()
    plt.plot(vols, label="Volume")
    plt.plot(smooth, label="Smoothed")
    plt.scatter(peaks, smooth[peaks], label="ED")
    plt.scatter(valleys, smooth[valleys], label="ES")
    plt.legend()
    plt.title("Volume Curve")
    plt.show()


# ------------------------------
# TEST INPUT (IMPORTANT)
# ------------------------------
# Create masks from your dataset
frames = [dataset[i][0] for i in range(30)]  # 30 frames

masks = []
for img in frames:
    x = img.astype(np.float32)
    if x.max() > 1:
        x = x / 255.0

    x = torch.tensor(x).unsqueeze(0).to(device)

    with torch.no_grad():
        pred = torch.sigmoid(model(x))[0][0].cpu().numpy()

    masks.append((pred > 0.5).astype(np.uint8))

print("Total masks:", len(masks))

# ------------------------------
# RUN
# ------------------------------
run_ef_pipeline(masks)

In [ ]:
import numpy as np
import torch
import matplotlib.pyplot as plt
from scipy.ndimage import gaussian_filter1d
from scipy.signal import find_peaks

# ------------------------------
# STEP 1: GET FRAMES
# ------------------------------
frames = [dataset[i][0] for i in range(80)]  # increase frames

# ------------------------------
# STEP 2: PREDICT MASKS
# ------------------------------
masks = []

for img in frames:
    x = img.astype(np.float32)
    if x.max() > 1:
        x = x / 255.0

    x = torch.tensor(x).unsqueeze(0).to(device)

    with torch.no_grad():
        pred = torch.sigmoid(model(x))[0][0].cpu().numpy()

    masks.append((pred > 0.5).astype(np.uint8))

print("Total masks:", len(masks))

# ------------------------------
# STEP 3: VOLUME
# ------------------------------
def compute_volume(mask, pixel_spacing=0.1):
    volume = 0
    for row in mask:
        width = np.sum(row > 0)
        if width > 0:
            r = (width * pixel_spacing) / 2
            volume += np.pi * r * r * pixel_spacing
    return volume

vols = np.array([compute_volume(m) for m in masks])

print("Volumes sample:", vols[:10])

# ------------------------------
# STEP 4: SMOOTH + FIND PEAKS
# ------------------------------
smooth = gaussian_filter1d(vols, sigma=2)

peaks, _ = find_peaks(smooth, distance=5)
valleys, _ = find_peaks(-smooth, distance=5)

print("Peaks:", peaks)
print("Valleys:", valleys)

# ------------------------------
# STEP 5: EF
# ------------------------------
def compute_ef(volumes, peaks, valleys):
    ef_values = []
    for ed, es in zip(peaks, valleys):
        EDV = volumes[ed]
        ESV = volumes[es]
        if EDV > 0:
            ef = (EDV - ESV) / EDV * 100
            ef_values.append(ef)
    return ef_values

ef_values = compute_ef(smooth, peaks, valleys)

print("Raw EF:", ef_values)

# ------------------------------
# STEP 6: FALLBACK (IMPORTANT)
# ------------------------------
if len(ef_values) == 0:
    print("⚠️ No cycles detected → using fallback")

    EDV = np.max(smooth)
    ESV = np.min(smooth)

    final_ef = (EDV - ESV) / EDV * 100
    print("Fallback EF:", final_ef)

else:
    final_ef = np.median(ef_values)
    print("Final EF:", final_ef)

# ------------------------------
# STEP 7: PLOT
# ------------------------------
plt.figure(figsize=(8,4))
plt.plot(vols, label="Volume")
plt.plot(smooth, label="Smoothed")
plt.scatter(peaks, smooth[peaks], label="ED")
plt.scatter(valleys, smooth[valleys], label="ES")
plt.legend()
plt.title("Volume Curve")
plt.show()

In [ ]:
# ==============================
# FINAL GRAD-CAM (SHARP + FOCUSED)
# ==============================
import torch
import numpy as np
import cv2
import matplotlib.pyplot as plt

model.eval()

# ------------------------------
# STORAGE
# ------------------------------
activations = []
gradients = []

def forward_hook(module, input, output):
    activations.clear()
    activations.append(output)

def backward_hook(module, grad_input, grad_output):
    gradients.clear()
    gradients.append(grad_output[0])

# ------------------------------
# TARGET LAYER (deep features)
# ------------------------------
target_layer = model.middle

# Register hooks (fixed)
target_layer.register_forward_hook(forward_hook)
target_layer.register_full_backward_hook(backward_hook)

# ------------------------------
# INPUT
# ------------------------------
img, mask = dataset[0]

x = img.astype(np.float32)
if x.max() > 1:
    x = x / 255.0

x_tensor = torch.tensor(x).unsqueeze(0).to(device)

# ------------------------------
# FORWARD
# ------------------------------
output = model(x_tensor)

# ------------------------------
# SEGMENTATION-FOCUSED LOSS
# ------------------------------
pred_mask = torch.sigmoid(output)

# Strong focus on confident regions
loss = (pred_mask ** 2).mean()

# ------------------------------
# BACKWARD
# ------------------------------
model.zero_grad()
loss.backward()

# ------------------------------
# BUILD CAM
# ------------------------------
act = activations[0].detach().cpu().numpy()[0]
grad = gradients[0].detach().cpu().numpy()[0]

weights = np.mean(grad, axis=(1,2))

cam = np.zeros(act.shape[1:], dtype=np.float32)

for i, w in enumerate(weights):
    cam += w * act[i]

# ReLU
cam = np.maximum(cam, 0)

# Normalize
cam -= cam.min()
cam /= (cam.max() + 1e-8)

# Resize + smooth (IMPROVED)
cam = cv2.resize(cam, (x.shape[2], x.shape[1]), interpolation=cv2.INTER_LINEAR)
cam = cv2.GaussianBlur(cam, (7,7), 0)

# ------------------------------
# HEATMAP
# ------------------------------
heatmap = cv2.applyColorMap(np.uint8(255 * cam), cv2.COLORMAP_JET)

img_vis = (x.transpose(1,2,0) * 255).astype(np.uint8)

overlay = cv2.addWeighted(img_vis, 0.6, heatmap, 0.4, 0)

# ------------------------------
# SEGMENTATION OUTPUT
# ------------------------------
pred_binary = (pred_mask[0][0].detach().cpu().numpy() > 0.5).astype(np.uint8)

# ------------------------------
# PLOT
# ------------------------------
plt.figure(figsize=(18,4))

plt.subplot(1,5,1)
plt.title("Original")
plt.imshow(img_vis)
plt.axis('off')

plt.subplot(1,5,2)
plt.title("GT Mask")
plt.imshow(mask[0] if len(mask.shape)==3 else mask, cmap='gray')
plt.axis('off')

plt.subplot(1,5,3)
plt.title("Prediction")
plt.imshow(pred_binary, cmap='gray')
plt.axis('off')

plt.subplot(1,5,4)
plt.title("Grad-CAM")
plt.imshow(cam, cmap='jet')
plt.axis('off')

plt.subplot(1,5,5)
plt.title("Overlay")
plt.imshow(overlay)
plt.axis('off')

plt.show()

In [ ]:
import torch

torch.save(model.state_dict(), "model.pth")
print("Model saved!")

In [ ]:
import shutil

shutil.make_archive("project", 'zip', "/kaggle/working")

In [ ]:
# ==============================
# FULL TRAINING CELL (FIXED)
# ==============================
import torch
import torch.nn as nn
import torch.optim as optim

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

model = UNet().to(device)

# ------------------------------
# LOSS (BCE + DICE)
# ------------------------------
def dice_loss(pred, target):
    smooth = 1e-6
    pred = pred.view(-1)
    target = target.view(-1)
    intersection = (pred * target).sum()
    return 1 - (2 * intersection + smooth) / (pred.sum() + target.sum() + smooth)

def combined_loss(pred, target):
    bce = nn.BCEWithLogitsLoss()(pred, target)
    dice = dice_loss(torch.sigmoid(pred), target)
    return bce + dice

optimizer = optim.Adam(model.parameters(), lr=1e-3)

# ------------------------------
# TRAIN
# ------------------------------
EPOCHS = 15

for epoch in range(EPOCHS):
    model.train()
    total_loss = 0

    print(f"\n🚀 Epoch {epoch+1}")

    for i, (images, masks) in enumerate(train_loader):

        images = images.float().to(device)
        masks = masks.float().to(device)

        # ensure mask shape [B,1,H,W]
        if len(masks.shape) == 3:
            masks = masks.unsqueeze(1)

        optimizer.zero_grad()

        outputs = model(images)

        loss = combined_loss(outputs, masks)
        loss.backward()
        optimizer.step()

        total_loss += loss.item()

        if i % 50 == 0:
            print(f"Batch {i} Loss: {loss.item():.4f}")

    print(f"Epoch Loss: {total_loss:.4f}")

In [ ]:
# ==============================
# FULL FIXED TRAINING PIPELINE
# ==============================
import torch
import torch.nn as nn
import torch.optim as optim
import numpy as np

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

# ------------------------------
# MODEL (MUST MATCH YOUR .pth)
# ------------------------------
class ConvBlock(nn.Module):
    def __init__(self, in_c, out_c):
        super().__init__()
        self.net = nn.Sequential(
            nn.Conv2d(in_c, out_c, 3, padding=1),
            nn.ReLU(),
            nn.Conv2d(out_c, out_c, 3, padding=1),
            nn.ReLU()
        )

    def forward(self, x):
        return self.net(x)

class UNet(nn.Module):
    def __init__(self):
        super().__init__()

        self.down1 = ConvBlock(3, 64)
        self.down2 = ConvBlock(64, 128)

        self.pool = nn.MaxPool2d(2)

        self.middle = ConvBlock(128, 256)

        self.up1 = nn.ConvTranspose2d(256, 128, 2, stride=2)
        self.conv1 = ConvBlock(256, 128)

        self.up2 = nn.ConvTranspose2d(128, 64, 2, stride=2)
        self.conv2 = ConvBlock(128, 64)

        self.out = nn.Conv2d(64, 1, 1)

    def forward(self, x):
        c1 = self.down1(x)
        p1 = self.pool(c1)

        c2 = self.down2(p1)
        p2 = self.pool(c2)

        m = self.middle(p2)

        u1 = self.up1(m)
        u1 = torch.cat([u1, c2], dim=1)
        c3 = self.conv1(u1)

        u2 = self.up2(c3)
        u2 = torch.cat([u2, c1], dim=1)
        c4 = self.conv2(u2)

        return self.out(c4)

model = UNet().to(device)

# ------------------------------
# LOSS (FIXED)
# ------------------------------
def combined_loss(pred, target):
    pos_weight = torch.tensor([8.0]).to(device)  # 🔥 important
    bce = nn.BCEWithLogitsLoss(pos_weight=pos_weight)(pred, target)

    pred_sig = torch.sigmoid(pred)

    smooth = 1e-6
    intersection = (pred_sig * target).sum()
    dice = 1 - (2 * intersection + smooth) / (pred_sig.sum() + target.sum() + smooth)

    return bce + dice

# ------------------------------
# OPTIMIZER (LOW LR)
# ------------------------------
optimizer = optim.Adam(model.parameters(), lr=1e-4)

# ------------------------------
# TRAIN LOOP
# ------------------------------
EPOCHS = 15

for epoch in range(EPOCHS):
    model.train()
    total_loss = 0

    print(f"\n🚀 Epoch {epoch+1}")

    for i, (images, masks) in enumerate(train_loader):

        images = images.float().to(device)
        masks = masks.float().to(device)

        # normalize images
        images = images / 255.0

        # fix mask shape
        if len(masks.shape) == 3:
            masks = masks.unsqueeze(1)

        # normalize masks (IMPORTANT)
        if masks.max() > 1:
            masks = masks / 255.0

        # skip empty masks
        if masks.sum() == 0:
            continue

        optimizer.zero_grad()

        outputs = model(images)

        loss = combined_loss(outputs, masks)
        loss.backward()
        optimizer.step()

        total_loss += loss.item()

        if i % 50 == 0:
            print(f"Batch {i} Loss: {loss.item():.4f}")

    print(f"Epoch Loss: {total_loss:.4f}")

# ------------------------------
# SAVE MODEL
# ------------------------------
torch.save(model.state_dict(), "best_model.pth")
print("✅ Model saved as best_model.pth")

In [ ]:
model.eval()

img, mask = dataset[10]

img = img.astype("float32") / 255.0
img_tensor = torch.tensor(img).unsqueeze(0).to(device)

with torch.no_grad():
    pred = torch.sigmoid(model(img_tensor))[0][0].cpu().numpy()

print("Min:", pred.min())
print("Max:", pred.max())